# Q2: ASR Output Cleanup Pipeline

Builds two cleanup operations on raw ASR output:
- **a)** Hindi number normalization
- **b)** English word detection in Hindi text

Run after Q1 (needs pretrained whisper-small outputs).

## Setup & Imports

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.data_utils import normalize_hindi_text
from src.number_normalizer import HindiNumberNormalizer
from src.english_detector import EnglishWordDetector

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\rajee\OneDrive\Desktop\JoshTech tasks


## STEP 1: Generate Raw ASR Transcripts (Pretrained Whisper-Small)

In [2]:
print("=" * 60)
print("STEP 1: Generate Raw ASR Transcripts")
print("=" * 60)

# Load pretrained whisper-small
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_NAME = "openai/whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading pretrained {MODEL_NAME} on {device}...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Hindi", task="transcribe")

STEP 1: Generate Raw ASR Transcripts
Loading pretrained openai/whisper-small on cpu...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/audio_tokenizer_c

In [3]:
# Load segments from Q1 preprocessing
SEGMENTS_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "segments")
TRANS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "transcriptions")

print("Generating raw ASR transcripts on training data segments...")
raw_asr_pairs = []

import librosa

# Walk through all segment directories
for seg_dir in sorted(os.listdir(SEGMENTS_DIR)):
    seg_path = os.path.join(SEGMENTS_DIR, seg_dir)
    if not os.path.isdir(seg_path):
        continue
    
    for seg_file in sorted(os.listdir(seg_path)):
        if not seg_file.endswith('.wav'):
            continue
        
        audio_path = os.path.join(seg_path, seg_file)
        
        try:
            # Load and transcribe
            audio, sr = librosa.load(audio_path, sr=16000, mono=True)
            input_features = processor.feature_extractor(
                audio, sampling_rate=16000, return_tensors="pt"
            ).input_features.to(device)
            
            with torch.no_grad():
                predicted_ids = model.generate(
                    input_features, language="Hindi", task="transcribe", max_length=225
                )
            
            raw_asr = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]
            
            raw_asr_pairs.append({
                'audio_path': audio_path,
                'raw_asr': raw_asr,
                'segment_file': seg_file,
            })
        except Exception as e:
            print(f"  Error processing {seg_file}: {e}")
            continue

print(f"\n✓ Generated {len(raw_asr_pairs)} raw ASR transcripts")

Generating raw ASR transcripts on training data segments...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


✓ Generated 4929 raw ASR transcripts


In [4]:
# Save raw ASR pairs
raw_asr_df = pd.DataFrame(raw_asr_pairs)
raw_asr_path = os.path.join(PROJECT_ROOT, "results", "raw_asr_outputs.csv")
os.makedirs(os.path.dirname(raw_asr_path), exist_ok=True)
raw_asr_df.to_csv(raw_asr_path, index=False, encoding='utf-8-sig')
print(f"✓ Saved to {raw_asr_path}")

✓ Saved to c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\results\raw_asr_outputs.csv


## STEP 2: Hindi Number Normalization

In [5]:
print("=" * 60)
print("STEP 2: Hindi Number Normalization")
print("=" * 60)

normalizer = HindiNumberNormalizer(preserve_idioms=True)

# Process all raw ASR transcripts
print("\nProcessing number normalization on all transcripts...")
num_norm_results = []

for pair in tqdm(raw_asr_pairs[:100], desc="Normalizing numbers"):  # Sample first 100
    result = normalizer.normalize_with_annotations(pair['raw_asr'])
    result['audio_path'] = pair['audio_path']
    num_norm_results.append(result)

# Find examples
conversion_examples = [r for r in num_norm_results if any(
    a['type'] == 'number_conversion' for a in r['annotations']
)]
idiom_examples = [r for r in num_norm_results if any(
    a['type'] == 'preserved_idiom' for a in r['annotations']
)]

print(f"\n✓ Found {len(conversion_examples)} transcripts with number conversions")
print(f"  Found {len(idiom_examples)} transcripts with preserved idioms")

STEP 2: Hindi Number Normalization

Processing number normalization on all transcripts...


Normalizing numbers: 100%|██████████| 100/100 [00:00<00:00, 3937.87it/s]


✓ Found 8 transcripts with number conversions
  Found 0 transcripts with preserved idioms


In [6]:
# Print before/after examples
print("--- CORRECT CONVERSION EXAMPLES ---")
for i, ex in enumerate(conversion_examples[:5]):
    print(f"\n  Example {i+1}:")
    print(f"    Before: {ex['original'][:80]}")
    print(f"    After:  {ex['normalized'][:80]}")
    for ann in ex['annotations']:
        print(f"    → '{ann['original']}' → '{ann['converted']}' [{ann['type']}]")

print("\n--- EDGE CASE EXAMPLES (Preserved Idioms) ---")
for i, ex in enumerate(idiom_examples[:3]):
    print(f"\n  Example {i+1}:")
    print(f"    Text: {ex['original'][:80]}")
    for ann in ex['annotations']:
        if ann['type'] == 'preserved_idiom':
            print(f"    → Preserved: '{ann['original']}' (idiomatic usage)")

--- CORRECT CONVERSION EXAMPLES ---

  Example 1:
    Before:  अपको अपको अपको तो कुछ सवल में नहीं आरा के ख़ा पहाते हैं तो हमार की सीने सुनी नह
    After:  अपको अपको अपको तो कुछ सवल में नहीं आरा के ख़ा पहाते हैं तो हमार की सीने सुनी नही
    → 'एक' → '1' [number_conversion]

  Example 2:
    Before:  अप इस बाप यह एक बाथ नहीं बताना चाँँगा तो हमारा हम आमरे एक मित्र थे उंगे पास यह 
    After:  अप इस बाप यह 1 बाथ नहीं बताना चाँँगा तो हमारा हम आमरे 1 मित्र थे उंगे पास यह पहल
    → 'एक' → '1' [number_conversion]
    → 'एक' → '1' [number_conversion]

  Example 3:
    Before:  अप सब को एक अट्टा बुलाग, सरी किलाज को बाग बुलाग लिया गद्जूस ने पिक याजा वो सामन
    After:  अप सब को 1 अट्टा बुलाग, सरी किलाज को बाग बुलाग लिया गद्जूस ने पिक याजा वो सामने 
    → 'एक' → '1' [number_conversion]

  Example 4:
    Before:  अगर आगे आगे और एक मासुमची शरादत फीग तरीगे जाड़ा बड़ी इंशराद दीती विदि विदि विदि
    After:  अगर आगे आगे और 1 मासुमची शरादत फीग तरीगे जाड़ा बड़ी इंशराद दीती विदि विदि विदि व
    → 'एक' → '1'

## STEP 3: English Word Detection in Hindi Text

In [7]:
print("=" * 60)
print("STEP 3: English Word Detection in Hindi Text")
print("=" * 60)

detector = EnglishWordDetector()

# Process all raw ASR transcripts
print("\nDetecting English words in transcripts...")
en_detection_results = []

for pair in tqdm(raw_asr_pairs[:100], desc="Detecting English"):
    result = detector.analyze_transcript(pair['raw_asr'])
    result['audio_path'] = pair['audio_path']
    en_detection_results.append(result)

# Stats
transcripts_with_english = [r for r in en_detection_results if r['english_word_count'] > 0]
total_english_words = sum(r['english_word_count'] for r in en_detection_results)

print(f"\n✓ Results:")
print(f"  Transcripts with English words: {len(transcripts_with_english)}/{len(en_detection_results)}")
print(f"  Total English words detected: {total_english_words}")
print(f"  Avg English ratio per transcript: {np.mean([r['english_ratio'] for r in en_detection_results]):.3f}")

STEP 3: English Word Detection in Hindi Text

Detecting English words in transcripts...


Detecting English: 100%|██████████| 100/100 [00:00<00:00, 500.26it/s]


✓ Results:
  Transcripts with English words: 2/100
  Total English words detected: 2
  Avg English ratio per transcript: 0.001


In [8]:
# Print tagged transcript examples
print("--- TAGGED TRANSCRIPT EXAMPLES ---")
for i, ex in enumerate(transcripts_with_english[:5]):
    print(f"\n  Example {i+1}:")
    print(f"    Original: {ex['original'][:80]}")
    print(f"    Tagged:   {ex['tagged'][:80]}")
    for ew in ex['english_words'][:3]:
        print(f"    → '{ew['clean_word']}' = {ew['english']} [{ew['detection_method']}]")

--- TAGGED TRANSCRIPT EXAMPLES ---

  Example 1:
    Original:  अपने दोस्तो के साग फुत्मल के मेच लेज लागता तब ये जोर से आवाज आई किसीने गोल करने
    Tagged:   अपने दोस्तो के साग फुत्मल के मेच लेज लागता तब ये जोर से आवाज आई किसीने [EN]गोल[/
    → 'गोल' = goal [lookup_table]

  Example 2:
    Original:  अज भी अपनी तादा लकती है तो बताएगे अपने कोई स्कूल में कोई अपने कोई शररत की हूं क
    Tagged:   अज भी अपनी तादा लकती है तो बताएगे अपने कोई [EN]स्कूल[/EN] में कोई अपने कोई शररत 
    → 'स्कूल' = school [lookup_table]


## STEP 4: Combined Cleanup Pipeline

In [9]:
print("=" * 60)
print("STEP 4: Combined Cleanup Pipeline")
print("=" * 60)

def cleanup_pipeline(raw_text: str) -> dict:
    """Apply number normalization + English word detection."""
    # Step 1: Normalize numbers
    num_result = normalizer.normalize_with_annotations(raw_text)
    cleaned_text = num_result['normalized']
    
    # Step 2: Detect and tag English words
    en_result = detector.analyze_transcript(cleaned_text)
    
    return {
        'raw_input': raw_text,
        'after_number_norm': cleaned_text,
        'after_english_tag': en_result['tagged'],
        'number_annotations': num_result['annotations'],
        'english_words': en_result['english_words'],
    }

# Demo on a few examples
print("\n--- FULL PIPELINE DEMO ---")
demo_texts = [pair['raw_asr'] for pair in raw_asr_pairs[:5]]
for i, text in enumerate(demo_texts):
    result = cleanup_pipeline(text)
    print(f"\n  Example {i+1}:")
    print(f"    Raw:         {result['raw_input'][:70]}")
    print(f"    Num-cleaned: {result['after_number_norm'][:70]}")
    print(f"    EN-tagged:   {result['after_english_tag'][:70]}")

STEP 4: Combined Cleanup Pipeline

--- FULL PIPELINE DEMO ---

  Example 1:
    Raw:          अपने ताई बज्बागी स्वादे जीवन का सबसी सुनेरा वो बज्बागी स्वादे जीवन का
    Num-cleaned: अपने ताई बज्बागी स्वादे जीवन का सबसी सुनेरा वो बज्बागी स्वादे जीवन का 
    EN-tagged:   अपने ताई बज्बागी स्वादे जीवन का सबसी सुनेरा वो बज्बागी स्वादे जीवन का 

  Example 2:
    Raw:          अगर तो जमएदारिया भी नहीं हुती हैं और उगर उगर हुती हैं हुती हैं मस्ती 
    Num-cleaned: अगर तो जमएदारिया भी नहीं हुती हैं और उगर उगर हुती हैं हुती हैं मस्ती ब
    EN-tagged:   अगर तो जमएदारिया भी नहीं हुती हैं और उगर उगर हुती हैं हुती हैं मस्ती ब

  Example 3:
    Raw:          अपको जुकरना जुकरना जुकरना सर्प अब उसर्प ये काम आँँँँँँँँँँँँँँँँँँँँँ
    Num-cleaned: अपको जुकरना जुकरना जुकरना सर्प अब उसर्प ये काम आँँँँँँँँँँँँँँँँँँँँँँ
    EN-tagged:   अपको जुकरना जुकरना जुकरना सर्प अब उसर्प ये काम आँँँँँँँँँँँँँँँँँँँँँँ

  Example 4:
    Raw:          अगर बादिया वादिया वादिया वादिया वादिया वादिया वादिया वादिया वादिया वा
  

In [10]:
# Save combined results
combined_path = os.path.join(PROJECT_ROOT, "results", "q2_cleanup_results.csv")
combined_data = []
for pair in raw_asr_pairs[:100]:
    result = cleanup_pipeline(pair['raw_asr'])
    combined_data.append({
        'audio_path': pair['audio_path'],
        'raw_asr': result['raw_input'],
        'after_cleanup': result['after_english_tag'],
    })

pd.DataFrame(combined_data).to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f"✓ Results saved to {combined_path}")
print("\n✓ Q2 Complete!")

✓ Results saved to c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\results\q2_cleanup_results.csv

✓ Q2 Complete!
